In [ ]:
LEFT JOIN silver_rdm_session_status rdmstatus
  ON LOWER(TRIM(rdmstatus.session_status_src_name)) = LOWER(TRIM(COALESCE(apptatt.name, appt.status)))
 AND UPPER(rdmstatus.session_status_src_id) LIKE 'MPB%'

In [ ]:
%%sql
SELECT DISTINCT
    session_status_src_id,
    session_status_src_name,
    session_status_name_conformed
FROM silver_rdm_session_status
WHERE LOWER(session_status_src_id) LIKE 'mpb_%'
ORDER BY session_status_src_id;

In [ ]:
LEFT JOIN silver_rdm_session_status rdmstatus
  ON LOWER(TRIM(rdmstatus.session_status_src_id)) = LOWER(TRIM(CONCAT('MPB_', COALESCE(apptatt.name, appt.status))))

In [ ]:
As part of the Sessions Silver Table refactor, the MPB block was split into a separate source-specific section for isolated testing and validation. During this work, additional changes were required to align the logic with the new silver_rdm_session_status RDM table/schema. The previous implementation was joined using the old RDM structure and old status columns, but in the new RDM table both the join logic and the column structure have changed.
Specifically, the old join was based on the earlier status column approach, whereas the new RDM requires source-id based mapping using the updated source status field. In addition, the new RDM table contains renamed/restructured columns, so the old status-mapping references were no longer valid and had to be updated.
The MPB block was therefore updated to:

split the MPB logic into a separate source-specific block for controlled validation
replace the old session status join with the new join logic required by the updated RDM table
move from the previous join column approach to the new source-id based join
replace old status-related column references with the new RDM columns
update downstream status-derived fields to use the new conformed status column
apply typed NULL casting in the isolated test table where required to avoid schema-resolution issues during validation

After these changes, the MPB block was validated successfully and the expected output count was retained.

थोडं अजून polished title pair पण देऊ:

User Story:
Update Sessions Silver Table source blocks to align with new RDM Session Status schema

Task:
MPB block refactor and new RDM Session Status schema alignment

In [ ]:
LEFT JOIN silver_rdm_session_status rdmstatus
  ON LOWER(TRIM(rdmstatus.session_status_src_id)) =
     LOWER(TRIM(CONCAT('WIP_',
        CASE
            WHEN AE.activity_date_time > current_timestamp() THEN 'pending'
            WHEN serv.description LIKE ('%DNA%') THEN 'dna'
            WHEN serv.description LIKE ('%Cancel%') THEN 'cancelled'
            WHEN Acts.description IS NOT NULL THEN LOWER(TRIM(Acts.description))
            WHEN COBS.description IS NOT NULL THEN LOWER(TRIM(COBS.description))
            ELSE 'unknown'
        END
     )))

In [ ]:
Updated the WIP sessions block to support the new silver_rdm_session_status schema, replaced the old join with the new source-id based mapping, updated downstream conformed status logic, and validated the block successfully.


Changes completed:
Replaced the old RDM session status join with the new source-id based join
Removed dependency on the old source-system filter column in the old join logic
Replaced old status-mapping columns with the new conformed status column
Updated downstream session status logic accordingly
Applied typed NULL casting in the isolated test table where required to avoid schema resolution issues during validation
Successfully validated the WIP block after changes

In [ ]:
LEFT JOIN silver_rdm_session_status rdmstatus
  ON LOWER(TRIM(rdmstatus.session_status_src_id)) = LOWER(TRIM(CONCAT('SONE_', m.mapping)))

In [ ]:
I checked it. The warning is coming from the table schema rather than the select logic itself. The columns session_admin_duration_mins and session_derived_duration_mins were originally created using plain NULL, so Fabric has stored them as VOID type. Because of that, the notebook queries can still run, but the Fabric table preview shows a warning since VOID columns are not supported there.

We have already identified the fix pattern for this: those nullable columns need to be created using typed NULLs such as CAST(NULL AS INT). Once the main silver_sessions table is rebuilt with the corrected typed schema, the preview warning should clear as well.

In [ ]:
SELECT column_name, data_type
FROM INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'silver_sessions'
  AND column_name IN ('session_admin_duration_mins', 'session_derived_duration_mins');